# Notebook 3 - Inspeção de portas, aberturas e hospedeiros no IFC e no RDF

**CIV2155 | Análise-piloto da CQ2 | versão 1.0 | 28/10/2026**

**Pergunta:** por que oito das 53 portas não aparecem nas 45 cadeias parede-abertura-porta reconhecidas pelo critério inicial?

O procedimento seleciona entidades pela classe IFC, percorre relações declaradas e verifica se os pares hospedeiro-porta aparecem no RDF ampliado da arquitetura. Os arquivos IFC e RDF são lidos sem alteração. As tabelas e o registro de execução são exportados ao final.

**Origem do procedimento:** adaptação específica deste estudo das APIs do IfcOpenShell e RDFLib e das relações definidas pela buildingSMART e pela BOT. As fontes fundamentam as operações; a combinação das consultas e o filtro `IfcWall` são decisões deste ensaio.

**Contexto da conversão:** o RDF de entrada foi produzido anteriormente com IFCtoLBD 2.49.0, com a correção textual registrada no Notebook 2 e a opção `--hasGeometry=false`. Fundamentação do conversor: Bonduel et al. (2018), [artigo](https://ceur-ws.org/Vol-2159/04paper.pdf) e [repositório dos autores](https://github.com/jyrkioraskari/IFCtoLBD). A inspeção abaixo é uma etapa posterior à conversão.

**Limites:** não calcular geometria, associar portas a cômodos, reclassificar objetos ou resolver a integração entre disciplinas. Uma aresta direta não comprova a preservação de todo o IFC.

## Como executar

Executar as células na ordem apresentada. No Colab, a etapa 2 solicita o envio de **um IFC arquitetônico e um TTL arquitetônico ampliado**. No computador, abrir a partir da pasta raiz do projeto ou ajustar `PASTA_PROJETO` na etapa 2.

Entradas deste ensaio:

- `02_Dados_BIM/Arquitetura/TIPO2-ARQ-DTV_R03.ifc`.
- `04_Resultados/02_Diagnostico_Integracao/arquitetura/grafo.ttl`.

O TTL inicial com 411 triplas é outro recorte e não corresponde à entrada deste teste. Os hashes e a conferência da etapa 10 identificam a base utilizada.

## 1. Preparar as bibliotecas

**Operação:** verificar e, se necessário, instalar as versões usadas no teste. A instalação ocorre no ambiente Python em que o notebook está sendo executado.

**Fontes técnicas:** [R1 - exemplos oficiais do IfcOpenShell](https://docs.ifcopenshell.org/ifcopenshell-python/code_examples.html), [R2 - util.element](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/element/index.html) e [R3 - RDFLib: Navigating Graphs](https://rdflib.readthedocs.io/en/latest/intro_to_graphs/). Código das bibliotecas: [IfcOpenShell](https://github.com/IfcOpenShell/IfcOpenShell) e [RDFLib](https://github.com/RDFLib/rdflib).

As páginas de documentação podem acompanhar versões mais recentes; as versões de execução são fixadas e registradas abaixo.

In [1]:
import sys, subprocess, importlib.metadata
VERSOES = {"ifcopenshell": "0.8.4", "rdflib": "7.1.4"}
for pacote, versao in VERSOES.items():
    try:
        instalada = importlib.metadata.version(pacote)
    except importlib.metadata.PackageNotFoundError:
        instalada = None
    if instalada != versao:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--quiet",
            f"{pacote}=={versao}"
        ])

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, hashlib, json, platform, zipfile
import ifcopenshell
import ifcopenshell.util.element
import rdflib
from rdflib import Namespace, RDF
from IPython.display import display, HTML
import html

# Exibição de tabelas sem exigir uma biblioteca de análise adicional.
def mostrar_tabela(linhas, colunas=None):
    if not linhas:
        print("Nenhum registro para os critérios desta tabela.")
        return
    colunas = colunas or list(linhas[0])
    cabecalho = "".join(f"<th>{html.escape(c)}</th>" for c in colunas)
    corpo = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(linha.get(c, '')))}</td>" for c in colunas
    ) + "</tr>" for linha in linhas)
    display(HTML(f"<table border='1'><tr>{cabecalho}</tr>{corpo}</table>"))

print({p: importlib.metadata.version(p) for p in VERSOES})

{'ifcopenshell': '0.8.4', 'rdflib': '7.1.4'}


## 2. Selecionar os dois arquivos de entrada

**Operação:** localizar o IFC e o TTL do ensaio. No Colab, enviar os arquivos indicados acima. O modo `upload` é automático no Colab; o modo `pasta` é automático no computador. Para usar uma pasta montada no Drive, alterar `MODO_ENTRADA` para `pasta` e informar seu caminho em `PASTA_PROJETO` após montá-la.

**Fonte:** [R7 - Colab: arquivos locais e Drive](https://colab.research.google.com/notebooks/io.ipynb). A organização das pastas é uma configuração deste estudo.

In [2]:
try:
    import google.colab
    EM_COLAB = True
except ImportError:
    EM_COLAB = False
MODO_ENTRADA = "upload" if EM_COLAB else "pasta"
PASTA_PROJETO = Path.cwd()  # Ajustar se a pasta raiz do projeto for outra.

if MODO_ENTRADA == "upload":
    from google.colab import files
    enviados = files.upload()
    candidatos_ifc = [Path(n) for n in enviados if n.lower().endswith(".ifc")]
    candidatos_rdf = [Path(n) for n in enviados if n.lower().endswith(".ttl")]
    if len(candidatos_ifc) != 1 or len(candidatos_rdf) != 1:
        raise ValueError("Enviar exatamente um arquivo IFC e um arquivo TTL.")
    caminho_ifc, caminho_rdf = candidatos_ifc[0], candidatos_rdf[0]
    pasta_saida = Path.cwd() / "Resultados_Inspecao_CQ2"
    del enviados  # O conteúdo já foi gravado pelo Colab em arquivos locais.
elif MODO_ENTRADA == "pasta":
    caminho_ifc = PASTA_PROJETO / "02_Dados_BIM/Arquitetura/TIPO2-ARQ-DTV_R03.ifc"
    caminho_rdf = PASTA_PROJETO / "04_Resultados/02_Diagnostico_Integracao/arquitetura/grafo.ttl"
    pasta_saida = (PASTA_PROJETO / "04_Resultados/05_Analise_Piloto_CQ2"
                   / "Reproducao_Notebook_v01")
else:
    raise ValueError("MODO_ENTRADA deve ser 'pasta' ou 'upload'.")

for caminho in [caminho_ifc, caminho_rdf]:
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")
pasta_saida.mkdir(parents=True, exist_ok=True)
print("IFC:", caminho_ifc.name)
print("RDF:", caminho_rdf.name)

IFC: TIPO2-ARQ-DTV_R03.ifc
RDF: grafo.ttl


## 3. Registrar a procedência e ler os arquivos

**Operação:** calcular SHA-256 e carregar o IFC e o grafo. O hash identifica os bytes da entrada e permite verificar se foram preservados. Ele não demonstra qualidade semântica.

**Fontes:** [R1 - abertura do IFC](https://docs.ifcopenshell.org/ifcopenshell-python/code_examples.html), [R3 - leitura do RDF](https://rdflib.readthedocs.io/en/latest/intro_to_graphs/) e [R8 - hashlib/SHA-256](https://docs.python.org/3/library/hashlib.html).

In [3]:
def sha256(caminho):
    return hashlib.sha256(caminho.read_bytes()).hexdigest()

hash_antes = {"ifc": sha256(caminho_ifc), "rdf": sha256(caminho_rdf)}
modelo = ifcopenshell.open(str(caminho_ifc))
grafo = rdflib.Graph().parse(caminho_rdf, format="turtle")
print("Esquema IFC:", modelo.schema)
print("Triplas RDF:", len(grafo))
print(json.dumps(hash_antes, indent=2))

Esquema IFC: IFC4
Triplas RDF: 56961
{
  "ifc": "0975bf1394694eff8746e5e49232bdf950c36c37c858edb715c9e7bc8f65603a",
  "rdf": "b6b6c38fc13711db4f17d145dbc4efb0ff32c73c113fcdb992a25c2f43c2e4a3"
}


## 4. Identificar as portas pela classe IFC

**Critério:** selecionar `IfcDoor`, incluindo eventuais subtipos, com `by_type`. Ler `GlobalId`, `Name` e `is_a()` para identificar cada ocorrência. O nome textual não é o critério de seleção.

**Fonte:** [R1 - seleção por classe e leitura de atributos](https://docs.ifcopenshell.org/ifcopenshell-python/code_examples.html). O uso de `IfcDoor` define o recorte desta pergunta.

In [4]:
portas = sorted(modelo.by_type("IfcDoor"), key=lambda objeto: objeto.GlobalId)
inventario_portas = [{"GlobalId": p.GlobalId, "classe": p.is_a(), "nome": p.Name}
                     for p in portas]
print("Portas selecionadas:", len(portas))
print("GlobalIds distintos:", len({p.GlobalId for p in portas}))
mostrar_tabela(inventario_portas[:5])

Portas selecionadas: 53
GlobalIds distintos: 53


GlobalId,classe,nome
0$7sr1l_18mg3F8eebY0Nr,IfcDoor,PA4
08IKprlfH2WAuRZ4DmjH6w,IfcDoor,PM5
08juIWcUDAthHGZeLDYGNp,IfcDoor,PM3
0AXbNPPML1WPFzZGGjGw$P,IfcDoor,PM4
0ELPqLqwH5ixdZTn1yzbnw,IfcDoor,PO2


## 5. Percorrer porta → abertura → hospedeiro no IFC

**Critério:** a porta é `RelatedBuildingElement` em `IfcRelFillsElement`; a abertura é `RelatingOpeningElement`. Em `IfcRelVoidsElement`, essa abertura é `RelatedOpeningElement` e seu hospedeiro é `RelatingBuildingElement`.

São construídos índices pelo `id()` da entidade no próprio IFC. Todas as relações encontradas são conservadas. Havendo mais de um hospedeiro, a rotina não escolhe um silenciosamente. O filtro inicial aceita exatamente um preenchimento, um vazio e um hospedeiro da classe `IfcWall`.

**Fontes semânticas:** [R4a - IfcRelFillsElement](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/ADD2_TC1/HTML/schema/ifcproductextension/lexical/ifcrelfillselement.htm) e [R4b - IfcRelVoidsElement](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/ADD2_TC1/HTML/schema/ifcproductextension/lexical/ifcrelvoidselement.htm). **Fonte da leitura dos tipos e propriedades:** [R2 - get_type/get_psets](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/element/index.html). O filtro adicional `IfcWall` é o critério da consulta inicial deste estudo.

In [5]:
preenchimentos = defaultdict(list)
vazios = defaultdict(list)
for relacao in modelo.by_type("IfcRelFillsElement"):
    preenchimentos[relacao.RelatedBuildingElement.id()].append(relacao)
for relacao in modelo.by_type("IfcRelVoidsElement"):
    vazios[relacao.RelatedOpeningElement.id()].append(relacao)

casos_ifc = []
hospedeiros = {}
for porta in portas:
    preenchimento = preenchimentos[porta.id()]
    aberturas = [r.RelatingOpeningElement for r in preenchimento]
    relacoes_vazios = [r for abertura in aberturas for r in vazios[abertura.id()]]
    candidatos = {r.RelatingBuildingElement.id(): r.RelatingBuildingElement
                  for r in relacoes_vazios}
    hospedeiro = next(iter(candidatos.values())) if len(candidatos) == 1 else None
    tipo_porta = ifcopenshell.util.element.get_type(porta)
    aceito = (len(preenchimento) == 1 and len(relacoes_vazios) == 1
              and hospedeiro is not None and hospedeiro.is_a("IfcWall"))
    if hospedeiro is not None:
        tipo = ifcopenshell.util.element.get_type(hospedeiro)
        propriedades = ifcopenshell.util.element.get_psets(hospedeiro)
        hospedeiros[hospedeiro.GlobalId] = {
            "GlobalId": hospedeiro.GlobalId, "classe": hospedeiro.is_a(),
            "nome": hospedeiro.Name, "tipo": None if tipo is None else tipo.Name,
            "FNDE_Pedras": propriedades.get("FNDE", {}).get("Pedras")
        }
    linha = {
        "porta_guid": porta.GlobalId, "porta_nome": porta.Name,
        "porta_tipo": None if tipo_porta is None else tipo_porta.Name,
        "preenchimentos_ifc": len(preenchimento), "aberturas_ifc": len(aberturas),
        "vazios_ifc": len(relacoes_vazios),
        "abertura_guid": " | ".join(a.GlobalId for a in aberturas),
        "hospedeiro_guid": None if hospedeiro is None else hospedeiro.GlobalId,
        "hospedeiro_classe": None if hospedeiro is None else hospedeiro.is_a(),
        "hospedeiro_nome": None if hospedeiro is None else hospedeiro.Name,
        "aceito_no_criterio_IfcWall": aceito
    }
    casos_ifc.append((porta, hospedeiro, linha))
print("Classes dos hospedeiros:", dict(Counter(l["hospedeiro_classe"] for _, _, l in casos_ifc)))

Classes dos hospedeiros: {'IfcWall': 45, 'IfcFurniture': 8}


## 6. Examinar os casos fora do critério

**Operação:** exibir os GUIDs das portas excluídas pelo critério e as descrições dos hospedeiros. `FNDE/Pedras` e o nome do tipo são propriedades declaradas no arquivo, não conclusões obtidas por geometria.

**Fonte:** [R2 - get_type/get_psets](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/element/index.html). A definição de [R5 - IfcFurniture](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/ADD2_TC1/HTML/schema/ifcsharedfacilitieselements/lexical/ifcfurniture.htm) permite contextualizar a classe, mas não comprova sua adequação aos objetos deste projeto. Essa adequação não é julgada neste teste.

In [6]:
fora_ifc = [linha for _, _, linha in casos_ifc if not linha["aceito_no_criterio_IfcWall"]]
mostrar_tabela(fora_ifc, ["porta_guid", "porta_nome", "hospedeiro_guid", "hospedeiro_classe"])
ids_fora = sorted({r["hospedeiro_guid"] for r in fora_ifc if r["hospedeiro_guid"]})
mostrar_tabela([hospedeiros[guid] for guid in ids_fora])

porta_guid,porta_nome,hospedeiro_guid,hospedeiro_classe
08IKprlfH2WAuRZ4DmjH6w,PM5,0scVyEQ7H1WBwab8_01NcZ,IfcFurniture
0LsKoafCbDSe3m0_7bRcqA,PM5,3My5Kwz2j0Tx3VQDQdBCK$,IfcFurniture
0q4YSxByfFc8mvXtLdses8,PM5,3My5Kwz2j0Tx3VQDQdBCK$,IfcFurniture
12cC5Ioe952e0vutOtETR8,PM5,0scVyEQ7H1WBwab8_01NcZ,IfcFurniture
13$TzEYPL7BwyJuoALhFMi,PM5,0scVyEQ7H1WBwab8_01NcZ,IfcFurniture
1fc6gQjLr8S9y9TpDxmiO0,PM5,0scVyEQ7H1WBwab8_01NcZ,IfcFurniture
2mN81WWIr2Duv5Qp0diyV_,PM5,3My5Kwz2j0Tx3VQDQdBCK$,IfcFurniture
3DwHjxFVTCoRGzeEKYeWeg,PM5,3My5Kwz2j0Tx3VQDQdBCK$,IfcFurniture


GlobalId,classe,nome,tipo,FNDE_Pedras
0scVyEQ7H1WBwab8_01NcZ,IfcFurniture,DV1,EI-GRA-Divisória - Granito cinza Andorinha 30,DIVISÓRIA - GRANITO CINZA ANDORINHA - esp. 3 cm
3My5Kwz2j0Tx3VQDQdBCK$,IfcFurniture,DV1,EI-GRA-Divisória - Granito cinza Andorinha 30,DIVISÓRIA - GRANITO CINZA ANDORINHA - esp. 3 cm


## 7. Localizar as entidades correspondentes no RDF pelo GlobalId

**Critério:** percorrer o predicado `props:globalIdIfcRoot_attribute_simple` e construir um índice de GlobalId para recurso RDF. O predicado é o encontrado na saída IFCtoLBD deste ensaio; não é um predicado universal de todo RDF.

**Fonte técnica:** [R3 - busca por padrões de triplas, seção Basic Triple Matching](https://rdflib.readthedocs.io/en/latest/intro_to_graphs/#basic-triple-matching). O termo `None` funciona como curinga. **Procedência do formato de entrada:** [IFCtoLBD, código dos autores](https://github.com/jyrkioraskari/IFCtoLBD).

In [7]:
PROPS = Namespace("http://lbd.arch.rwth-aachen.de/props#")
BOT = Namespace("https://w3id.org/bot#")
indice = defaultdict(set)
for sujeito, _, valor in grafo.triples(
        (None, PROPS.globalIdIfcRoot_attribute_simple, None)):
    indice[str(valor)].add(sujeito)
print("Valores de GlobalId indexados:", len(indice))

Valores de GlobalId indexados: 2116


## 8. Conferir as arestas diretas entre hospedeiro e porta

**Critério:** exigir correspondência única para os dois GlobalIds e procurar qualquer predicado direto nos dois sentidos. Registrar os predicados e as triplas encontradas. Não se impõe previamente que o resultado seja `bot:hasSubElement`.

**Fontes:** [R3 - Graph.triples, Graph.objects e RDF.type](https://rdflib.readthedocs.io/en/latest/intro_to_graphs/); [R6 - BOT/hasSubElement](https://w3c-lbd-cg.github.io/bot/#hasSubElement), relação que descreve hospedagem ou composição entre elementos. Esta célula não utiliza SPARQL nem materializa inferências.

In [8]:
registros = []
detalhes = []
for porta, hospedeiro, linha_ifc in casos_ifc:
    portas_rdf = indice[porta.GlobalId]
    hosts_rdf = indice[hospedeiro.GlobalId] if hospedeiro is not None else set()
    relacoes = []
    if len(portas_rdf) == 1 and len(hosts_rdf) == 1:
        p = next(iter(portas_rdf))
        h = next(iter(hosts_rdf))
        relacoes = list(grafo.triples((h, None, p))) + list(grafo.triples((p, None, h)))
    linha = dict(linha_ifc)
    linha.update({
        "correspondencia_rdf_unica": len(portas_rdf) == 1 and len(hosts_rdf) == 1,
        "arestas_diretas_rdf": len(relacoes),
        "predicados_rdf": " | ".join(sorted({str(t[1]) for t in relacoes}))
    })
    registros.append(linha)
    detalhes.append({
        **linha, "triplas_rdf": [[str(v) for v in t] for t in relacoes],
        "tipos_rdf_porta": [str(x) for s in portas_rdf for x in grafo.objects(s, RDF.type)],
        "tipos_rdf_hospedeiro": [str(x) for s in hosts_rdf for x in grafo.objects(s, RDF.type)]
    })
fora_criterio = [r for r in registros if not r["aceito_no_criterio_IfcWall"]]
mostrar_tabela(fora_criterio, ["porta_guid", "hospedeiro_classe", "correspondencia_rdf_unica", "arestas_diretas_rdf", "predicados_rdf"])

porta_guid,hospedeiro_classe,correspondencia_rdf_unica,arestas_diretas_rdf,predicados_rdf
08IKprlfH2WAuRZ4DmjH6w,IfcFurniture,True,1,https://w3id.org/bot#hasSubElement
0LsKoafCbDSe3m0_7bRcqA,IfcFurniture,True,1,https://w3id.org/bot#hasSubElement
0q4YSxByfFc8mvXtLdses8,IfcFurniture,True,1,https://w3id.org/bot#hasSubElement
12cC5Ioe952e0vutOtETR8,IfcFurniture,True,1,https://w3id.org/bot#hasSubElement
13$TzEYPL7BwyJuoALhFMi,IfcFurniture,True,1,https://w3id.org/bot#hasSubElement
1fc6gQjLr8S9y9TpDxmiO0,IfcFurniture,True,1,https://w3id.org/bot#hasSubElement
2mN81WWIr2Duv5Qp0diyV_,IfcFurniture,True,1,https://w3id.org/bot#hasSubElement
3DwHjxFVTCoRGzeEKYeWeg,IfcFurniture,True,1,https://w3id.org/bot#hasSubElement


## 9. Consolidar as observações

**Operação:** contar os casos efetivamente obtidos. O código de busca não recebe os números 53, 45 e 8 como condição para selecionar entidades. Uma porta fora do filtro não é automaticamente uma falha do conversor.

**Fontes dos critérios:** R4a/R4b (relações IFC) e R6 (semântica BOT). As contagens e a comparação do filtro são resultados deste estudo.

In [9]:
resumo = {
    "portas_ifc": len(registros),
    "cadeias_com_hospedeiro_unico": sum(r["hospedeiro_guid"] is not None for r in registros),
    "por_classe_hospedeiro": dict(Counter(r["hospedeiro_classe"] for r in registros)),
    "aceitas_criterio_IfcWall": sum(r["aceito_no_criterio_IfcWall"] for r in registros),
    "fora_criterio_IfcWall": len(fora_criterio),
    "fora_criterio_com_aresta_rdf": sum(r["arestas_diretas_rdf"] > 0 for r in fora_criterio),
    "todas_com_aresta_rdf": sum(r["arestas_diretas_rdf"] > 0 for r in registros),
    "hospedeiros_fora_criterio": len({r["hospedeiro_guid"] for r in fora_criterio})
}
print(json.dumps(resumo, indent=2, ensure_ascii=False))
predicados = Counter(p for r in registros for p in r["predicados_rdf"].split(" | ") if p)
print("Predicados observados:", dict(predicados))

{
  "portas_ifc": 53,
  "cadeias_com_hospedeiro_unico": 53,
  "por_classe_hospedeiro": {
    "IfcWall": 45,
    "IfcFurniture": 8
  },
  "aceitas_criterio_IfcWall": 45,
  "fora_criterio_IfcWall": 8,
  "fora_criterio_com_aresta_rdf": 8,
  "todas_com_aresta_rdf": 53,
  "hospedeiros_fora_criterio": 2
}
Predicados observados: {'https://w3id.org/bot#hasSubElement': 53}


## 10. Verificar a reprodução do caso relatado

**Operação:** comparar a identidade das entradas e os indicadores obtidos com o registro anterior. Esta conferência é posterior à busca. Se as entradas forem diferentes, a rotina informa que os resultados não são diretamente comparáveis ao relatório.

Os hashes e indicadores abaixo procedem do `Registro_Inspecao_Portas_CQ2_v01.json` arquivado no projeto. O sucesso desta conferência não valida toda a conversão ou a integração. **Fonte técnica do hash:** [R8 - hashlib](https://docs.python.org/3/library/hashlib.html).

In [10]:
hash_referencia = {
    "ifc": "0975bf1394694eff8746e5e49232bdf950c36c37c858edb715c9e7bc8f65603a",
    "rdf": "b6b6c38fc13711db4f17d145dbc4efb0ff32c73c113fcdb992a25c2f43c2e4a3"
}
resumo_referencia = {
    "portas_ifc": 53, "cadeias_com_hospedeiro_unico": 53,
    "por_classe_hospedeiro": {"IfcWall": 45, "IfcFurniture": 8},
    "aceitas_criterio_IfcWall": 45, "fora_criterio_IfcWall": 8,
    "fora_criterio_com_aresta_rdf": 8, "todas_com_aresta_rdf": 53,
    "hospedeiros_fora_criterio": 2
}
mesma_base = hash_antes == hash_referencia
if mesma_base:
    assert resumo == resumo_referencia, "Os indicadores divergem do registro anterior."
    print("Mesmas entradas; todos os indicadores reproduzem o teste anterior.")
else:
    print("Entradas diferentes: não afirmar reprodução do caso do relatório.")

Mesmas entradas; todos os indicadores reproduzem o teste anterior.


## 11. Salvar os registros e conferir a integridade das entradas

**Operação:** exportar inventário completo, subconjunto fora do critério e JSON com as relações observadas, as versões, os hashes e os limites. Criar também um ZIP para transferência. Os resultados ficam em uma subpasta própria de reprodução.

**Fontes técnicas:** [R8 - hashlib](https://docs.python.org/3/library/hashlib.html), [R9a - csv](https://docs.python.org/3/library/csv.html), [R9b - json](https://docs.python.org/3/library/json.html) e [R9c - zipfile](https://docs.python.org/3/library/zipfile.html). A escolha dos campos de auditoria é uma decisão deste estudo.

In [11]:
hash_depois = {"ifc": sha256(caminho_ifc), "rdf": sha256(caminho_rdf)}
assert hash_antes == hash_depois, "Algum arquivo de entrada foi alterado."
limites = [
    "Não julga a correção da classificação no projeto de autoria.",
    "Não calcula geometria nem determina o ambiente físico das portas.",
    "Não verifica a correspondência das aberturas no RDF.",
    "Não verifica caminhos indiretos nem materializa inferências.",
    "Uma aresta direta não comprova preservação integral do IFC ou integração."
]
payload = {
    "versao_notebook": "1.0", "execucao_utc": datetime.now(timezone.utc).isoformat(),
    "bibliotecas": {p: importlib.metadata.version(p) for p in VERSOES},
    "python": platform.python_version(), "mesma_base_do_relatorio": mesma_base,
    "arquivos": {k: {"nome": caminho.name, "sha256_antes": hash_antes[k],
                      "sha256_depois": hash_depois[k]}
                 for k, caminho in [("ifc", caminho_ifc), ("rdf", caminho_rdf)]},
    "resumo": resumo, "hospedeiros": hospedeiros, "portas": detalhes,
    "limites": limites
}
campos = list(registros[0]) if registros else ["porta_guid"]
saidas = []
for nome, linhas in [("Inventario_Portas_Notebook_v01.csv", registros),
                     ("Portas_Fora_Criterio_IfcWall_Notebook_v01.csv", fora_criterio)]:
    arquivo = pasta_saida / nome
    with arquivo.open("w", encoding="utf-8-sig", newline="") as fluxo:
        escritor = csv.DictWriter(fluxo, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(linhas)
    saidas.append(arquivo)
arquivo_json = pasta_saida / "Registro_Inspecao_Notebook_CQ2_v01.json"
arquivo_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf8")
saidas.append(arquivo_json)
arquivo_zip = pasta_saida / "Resultados_Notebook_CQ2_v01.zip"
with zipfile.ZipFile(arquivo_zip, "w", compression=zipfile.ZIP_DEFLATED) as pacote:
    for arquivo in saidas:
        pacote.write(arquivo, arcname=arquivo.name)
print("Entradas preservadas. Arquivos gerados:")
for arquivo in saidas + [arquivo_zip]:
    print(arquivo.name)

Entradas preservadas. Arquivos gerados:


Inventario_Portas_Notebook_v01.csv
Portas_Fora_Criterio_IfcWall_Notebook_v01.csv
Registro_Inspecao_Notebook_CQ2_v01.json
Resultados_Notebook_CQ2_v01.zip


## Interpretação do caso e limite da conclusão

Com as mesmas entradas do relatório, o teste identifica 53 portas: 45 com hospedeiro `IfcWall` e oito com hospedeiro `IfcFurniture`. Os dois hospedeiros desse último grupo são descritos no IFC como divisórias de granito. Os 53 pares hospedeiro-porta têm aresta direta no RDF, com o predicado `bot:hasSubElement`.

O resultado 45/53 representa a cobertura de um filtro restrito a `IfcWall`. A ausência dos oito casos nesse subconjunto não demonstra perda na conversão. A relação de cada elemento com ambientes físicos permanece sem demonstração neste teste.

**Questão metodológica a explicitar:** a pergunta pretende recuperar somente paredes da classe `IfcWall`, ou também divisórias e outros hospedeiros? A finalidade e as perguntas de competência orientam o escopo da ontologia: Noy e McGuinness (2001), *Ontology Development 101*, p. 5, [PDF dos autores](https://protege.stanford.edu/publications/ontology_development/ontology101.pdf). Essa referência fundamenta a definição da pergunta; não é a origem do código de inspeção.

**No Colab:** os resultados exportados ficam na pasta `Resultados_Inspecao_CQ2`, acessível pelo painel Arquivos. Baixar o ZIP e salvar também a cópia executada do notebook para conservar suas saídas.

## Referências e localização das operações

Consulta em 28 out. 2026. Todas as fontes técnicas são documentação dos próprios projetos ou a especificação do esquema/ontologia.

| ID | Fonte e trecho utilizado | Etapas |
|---|---|---|
| R1 | **IfcOpenShell. Code examples.** Seleção com `by_type`, atributos, leitura de tipos/propriedades. [Documentação](https://docs.ifcopenshell.org/ifcopenshell-python/code_examples.html); [código-fonte](https://github.com/IfcOpenShell/IfcOpenShell). | 1, 3, 4 |
| R2 | **IfcOpenShell. util.element.** `get_type` e `get_psets`. [Documentação](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/element/index.html). | 5, 6 |
| R3 | **RDFLib. Navigating Graphs.** *Basic Triple Matching*: `triples`, `None` como curinga, `objects`. [Documentação](https://rdflib.readthedocs.io/en/latest/intro_to_graphs/); [código-fonte](https://github.com/RDFLib/rdflib). | 1, 3, 7, 8 |
| R4a | **buildingSMART. IFC4 ADD2 TC1: IfcRelFillsElement.** Atributos e exemplo parede-abertura-porta. [Especificação](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/ADD2_TC1/HTML/schema/ifcproductextension/lexical/ifcrelfillselement.htm). | 5 |
| R4b | **buildingSMART. IFC4 ADD2 TC1: IfcRelVoidsElement.** Atributos `RelatingBuildingElement` e `RelatedOpeningElement`. [Especificação](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/ADD2_TC1/HTML/schema/ifcproductextension/lexical/ifcrelvoidselement.htm). | 5 |
| R5 | **buildingSMART. IFC4 ADD2 TC1: IfcFurniture.** Definição da classe observada. [Especificação](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/ADD2_TC1/HTML/schema/ifcsharedfacilitieselements/lexical/ifcfurniture.htm). | 6 |
| R6 | **Linked Building Data Community Group. Building Topology Ontology.** `hasSubElement`: hospedagem ou composição. [Especificação](https://w3c-lbd-cg.github.io/bot/#hasSubElement). | 8, 9 |
| R7 | **Google Colaboratory. External data: Local Files, Drive, Sheets, and Cloud Storage.** Upload de arquivos. [Notebook oficial](https://colab.research.google.com/notebooks/io.ipynb). | 2 |
| R8 | **Python Software Foundation. hashlib.** `sha256`. [Documentação](https://docs.python.org/3/library/hashlib.html). | 3, 10, 11 |
| R9 | **Python Software Foundation.** [csv](https://docs.python.org/3/library/csv.html), [json](https://docs.python.org/3/library/json.html), [zipfile](https://docs.python.org/3/library/zipfile.html). Exportação de registros. | 11 |

**Contexto científico do conversor:** BONDUEL, Mathias et al. *The IFC to Linked Building Data Converter: Current Status*. LDAC, CEUR Workshop Proceedings, v. 2159, p. 34-43, 2018. [Artigo](https://ceur-ws.org/Vol-2159/04paper.pdf); [repositório dos autores](https://github.com/jyrkioraskari/IFCtoLBD).

**Definição de escopo e perguntas:** NOY, Natalya F.; McGUINNESS, Deborah L. *Ontology Development 101: A Guide to Creating Your First Ontology*. Stanford, 2001, p. 5. [PDF](https://protege.stanford.edu/publications/ontology_development/ontology101.pdf).